# Shor's Algorithm

Shor's algorithm is used to factor an integer $N$ (find nontrivial divisors) in polynomial time with the help of a quantum computer. Classically, factoring a large $N$ is difficult and time-consuming, but Shor's algorithm reduces the problem to finding the order.

### Main idea behind the algorithm
We choose a random $a$, where $1<a<N$, and $gcd(a,N)=1$.  
The goal is to find the smallest $r$ such that
$$
a^r\equiv1\pmod N.
$$

If $r$ is even and $a^{r/2}\not\equiv -1\pmod N,$ then we obtain nontrivial divisors:
$$
p=\gcd(a^{r/2}-1,N),\qquad
q=\gcd(a^{r/2}+1,N).
$$


### Core of the algorithm

We define the unitary operator
$$
U_a|y\rangle=|ay\bmod N\rangle.
$$

This operator has eigenstates $|u_s\rangle$, $s=0,\dots,r-1$, with
$$
U_a|u_s\rangle=e^{2\pi i s/r}|u_s\rangle.
$$

Thus, the eigenphases are
$$
\phi_s=\frac{s}{r}.
$$

Using QPE, we can estimate these phases and obtain information about $r$.

### Construction of eigenstates
The eigenstates can be expressed as:
$$
|u_s\rangle=
\frac{1}{\sqrt r}\sum_{k=0}^{r-1}
e^{-2\pi i sk/r}\,|a^k \bmod N\rangle,
\qquad s=0,\dots,r-1.
$$

To verify that this is indeed an eigenstate, we apply $U\equiv U_a$:
$$
U|u_s\rangle=
U\frac1{\sqrt r}\sum_{k=0}^{r-1}
e^{-2\pi i sk/r}|a^k\bmod N\rangle
=
\frac1{\sqrt r}\sum_{k=0}^{r-1}
e^{-2\pi i sk/r}U|a^k\bmod N\rangle.
$$

Since $U|a^k\bmod N\rangle=|a^{k+1}\bmod N\rangle,$ we obtain
$$
U|u_s\rangle=
\frac1{\sqrt r}\sum_{k=0}^{r-1}
e^{-2\pi i sk/r}|a^{k+1}\bmod N\rangle.
$$

We factor out the global phase using
$$
e^{-2\pi i sk/r}=e^{2\pi i s/r}\,e^{-2\pi i s(k+1)/r}.
$$
Therefore,
$$
U|u_s\rangle
=
e^{2\pi i s/r}\,
\frac1{\sqrt r}\sum_{k=0}^{r-1}
e^{-2\pi i s(k+1)/r}|a^{k+1}\bmod N\rangle.
$$

We rename the index $k+1=j$ (mod $r$):
$$
U|u_s\rangle
=
e^{2\pi i s/r}\,
\frac1{\sqrt r}\sum_{j=0}^{r-1}
e^{-2\pi i sj/r}|a^j\bmod N\rangle
=
e^{2\pi i s/r}|u_s\rangle.
$$

Thus:
$$
U|u_s\rangle=e^{2\pi i s/r}|u_s\rangle.
$$

### Phase calculation

Assume that the phase register contains $t$ qubits, and the auxiliary register initially is in the state $|1\rangle$.

Initial state:
$$
|\psi_0\rangle = |0\rangle^{\otimes t}\otimes |1\rangle.
$$

After applying Hadamard gates to the phase register, we obtain a superposition:
$$
|\psi_1\rangle = \frac{1}{2^{t/2}}\sum_{x=0}^{2^t-1}|x\rangle\otimes |1\rangle.
$$

We use the identity:
$$
|1\rangle = \frac{1}{\sqrt r}\sum_{s=0}^{r-1}|u_s\rangle,
$$
then we can express

$$
|\psi_1\rangle
=
\frac{1}{\sqrt r}\sum_{s=0}^{r-1}
\left(
\frac{1}{2^{t/2}}\sum_{x=0}^{2^t-1}|x\rangle
\right)\otimes |u_s\rangle.
$$

After the controlled $U_a^{2^k}$ operations:

$$
|x\rangle|u_s\rangle \mapsto e^{2\pi i x s/r}|x\rangle|u_s\rangle,
$$

and therefore

$$
|\psi_2\rangle
=
\frac{1}{\sqrt r}\sum_{s=0}^{r-1}
\left(
\frac{1}{2^{t/2}}\sum_{x=0}^{2^t-1}e^{2\pi i x s/r}|x\rangle
\right)\otimes |u_s\rangle.
$$

After the inverse QFT on the phase register, measurement gives $m$, where
$$
\frac{m}{2^t}\approx \frac{s}{r}.
$$

Then we use continued fractions to find a rational approximation:
$$
\frac{s'}{r'},\qquad r'<N.
$$

After that, the candidate $r'$ is checked by substituting it into $a^{r'}\equiv 1 \pmod N\;$

If the check fails, we use the next convergent or perform a new measurement.

In [5]:
import numpy as np
from math import gcd, ceil, log2
from fractions import Fraction

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate, UnitaryGate
from qiskit_aer import AerSimulator


N = 15
a = 4
t = 8                      # count (phase) register qubits
n = max(4, ceil(log2(N)))  # work register qubits
shots = 8192



def make_Ua_gate(a, N, n):
    dim = 2 ** n  # number of states in the auxiliary register
    U = np.zeros((dim, dim), dtype=complex)  # create an empty matrix

    for y in range(dim):
        if y < N:
            y2 = (a * y) % N
        else:
            y2 = y
        U[y2, y] = 1.0  # mark the transition with 1

    return UnitaryGate(U, label=f"Ua({a},{N})")




def run_shor(N, a, t, n, shots=4096):
    if gcd(a, N) != 1:
        g = gcd(a, N)
        print(f"gcd(a,N) = {g} -> factors immediately: {g} and {N//g}")
        return
    if N > 2**n:
        raise ValueError(f"{N} does not fit into {n} qubits.")

    Q = 2 ** t

    qc = QuantumCircuit(t + n, t)
    count = list(range(t))  # count qubit indices
    work = list(range(t, t + n))  # work qubit indices

    # set work to |1>
    qc.x(work[0])

    # count superposition
    for q in count:
        qc.h(q)

    # controlled-U^(2^k)
    Ua = make_Ua_gate(a, N, n)
    U_pow = Ua
    for k in range(t):  # for count qubits
        qc.append(U_pow.control(1), [count[k]] + work)  # add control qubit
        U_pow = U_pow.power(2)

    # inverse QFT on count
    qc.append(QFTGate(t).inverse(), count)

    # measure count
    qc.measure(count, range(t))




    
    sim = AerSimulator()
    tqc = transpile(qc, sim, optimization_level=1)
    print(tqc.num_qubits)
    result = sim.run(tqc, shots=shots).result()
    counts = result.get_counts()

    # Take the most frequent measurement
    m_bits = max(counts, key=counts.get)
    m = int(m_bits, 2)
    
    # m/2^t
    x = m / Q
    
    # Approximate r from continued fractions
    r = Fraction(x).limit_denominator(N - 1).denominator
    
    print("m_bits =", m_bits, " : ", counts[m_bits])
    print("m =", m, "x =", x, "r =", r)
    print("a^r mod N =", pow(a, r, N))
    
    # Try to find factors
    if r % 2 == 0 and pow(a, r, N) == 1 and pow(a, r//2, N) != N - 1:
        p = gcd(pow(a, r//2) - 1, N)
        q = gcd(pow(a, r//2) + 1, N)
        print("Factors:", p, q)
    else:
        print("This time it did not work. Try again (or use a different a / larger t).")
    
    

run_shor(N, a, t, n, shots)

12
m_bits = 10000000  :  4098
m = 128 x = 0.5 r = 2
a^r mod N = 1
Factors: 3 5
